# Defining Tools with MCP

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_server.py`; the finished version lives in `cli-project-complete/mcp_server.py`. This notebook shows what to change and verifies it against the complete copy.

Building an MCP server is simple with the official Python SDK — `FastMCP` + decorators + type hints generate the JSON tool schemas for you. We add two tools to the in-memory document server: **read a document** and **edit a document** (find-and-replace).

## What to modify

**File:** `cli-project/mcp_server.py` (the starter — your edits stay local via the `skip-worktree` guard)

**1. Add the import:**
```python
from pydantic import Field
```

**2. Add the two tools:**
```python
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string.",
)
def read_document(
    doc_id: str = Field(description="Id of the document to read"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]


@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(
        description="The text to replace. Must match exactly, including whitespace"
    ),
    new_str: str = Field(
        description="The new text to insert in place of the old text"
    ),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```

## How it works

- **`@mcp.tool(...)`** registers a function as an MCP tool and **auto-generates the JSON schema** from the signature — no hand-written schema.
- **`Field(description=...)`** (Pydantic) supplies each parameter's description, which becomes part of the schema Claude sees.
- **`read_document`** returns a doc by id; **`edit_document`** does a `str.replace`. Both `raise ValueError` on an unknown `doc_id` — the SDK surfaces that as a tool error Claude can recover from.

**SDK win:** type hints + decorators replace verbose JSON schemas — less boilerplate, Pydantic validation for free, ordinary readable Python.

## Verify — list the server's registered tools

Imports the **complete** server and asks the `FastMCP` instance for its tools, confirming the decorators registered them with the right schemas. Importing is safe — `mcp.run()` is guarded behind `if __name__ == "__main__"`.

> Did the exercise in the starter and want to check *your* version? Set `folder = "cli-project"` below (after you've added the tools).

In [3]:
import sys
import os
from dotenv import find_dotenv

folder = "cli-project-complete"   # answer key; switch to "cli-project" to check your own edits
PROJECT = os.path.join(os.path.dirname(find_dotenv()), "building-with-the-claude-api", "07-mcp", folder)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import mcp_server

tools = await mcp_server.mcp.list_tools()
for t in tools:
    print(t.name + " - " + t.description)
    print("  input properties:", list(t.inputSchema.get("properties", {}).keys()))
    print()

read_doc_contents - Read the contents of a document and return it as a string.
  input properties: ['doc_id']

edit_document - Edit a document by replacing a string in the documents content with a new string
  input properties: ['doc_id', 'old_str', 'new_str']



## Exercise the logic (optional)

The registered tools are invoked by the MCP framework (which injects arguments), so we exercise the same logic against the module's in-memory `docs` dict to confirm behavior.

In [4]:
print("before:", mcp_server.docs["plan.md"])
mcp_server.docs["plan.md"] = mcp_server.docs["plan.md"].replace("plan", "roadmap")
print("after: ", mcp_server.docs["plan.md"])
mcp_server.docs["plan.md"] = mcp_server.docs["plan.md"].replace("roadmap", "plan")  # restore

before: The plan outlines the steps for the project's implementation.
after:  The roadmap outlines the steps for the project's implementation.


## Where this leaves us

The **server** now exposes `read_doc_contents` and `edit_document`. But the CLI chatbot still can't use them: the **client's** `list_tools` / `call_tool` return `[]` (wired in **Implementing a client**). So the app runs and answers general questions, but can't touch documents yet.

Next: **The server inspector** — a GUI to connect to the server and invoke these tools directly, *without* the client.